# Retrain after the detector fix (technical review §13)

**Before running:** Notebook settings -> **Accelerator: GPU T4 x2** (or P100), **Internet: On**.

**Attach these three Kaggle datasets** (Add Input -> Datasets):

| Upload | Contents |
|---|---|
| `fyp-code` | `fyp_code.zip` (rebuilt - it now contains `teams.py` + `preflight.py`) |
| `fyp-clips` | the `dataset/dataset/` tree: 4 class folders, 508 clips |
| `fyp-weights` | `volleyball_best.pt` (still needed - but only for the BALL now) |

The cells below auto-discover those paths, so the dataset slugs do not have to match exactly.

**Why this retrain is mandatory:** `mamba_checkpoint_v2.pt` was trained on CSVs extracted through the broken detector path (fine-tune for players, imgsz 640, geometric team split). The fixed pipeline feeds a different feature distribution - measured mean spacing ~196 cm now vs ~330 cm in the old CSVs - so serving the old checkpoint is a train/serve mismatch (same class as L6 and §12.8). It shows up as every window flagged ANOMALY at near-chance confidence.

## 1. Pinned environment

Versions are pinned deliberately. §12.4 records why: the BoT-SORT adapter feeds our detections into ultralytics' *internal* tracker classes, which are not a stable contract. An unpinned install silently changed `_split_detections()` mid-project and killed the extraction run minutes in.

In [ ]:
!pip -q install ultralytics==8.4.30 supervision==0.27.0.post2 lap pytest pyflakes

import torch, ultralytics, supervision, cv2
print('torch      :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY')
print('ultralytics:', ultralytics.__version__)
print('supervision:', supervision.__version__)
print('cv2        :', cv2.__version__)
assert torch.cuda.is_available(), 'Turn the GPU on: Settings -> Accelerator -> GPU'

## 2. Locate inputs and unpack the code

In [ ]:
import os, glob, zipfile, shutil, sys

IN = '/kaggle/input'
WORK = '/kaggle/working'

def find_file(pattern):
    hits = glob.glob(f'{IN}/**/{pattern}', recursive=True)
    return sorted(hits, key=len)[0] if hits else None

CODE_ZIP = find_file('fyp_code.zip')
BALL_W   = find_file('volleyball_best.pt')

# The clip tree is whichever directory contains the four class folders.
CLASSES = ['Coordinated_Attack', 'Coordinated_Defense',
           'Delayed_Support', 'Spacing_Breakdown']
CLIPS = None
for root, dirs, _ in os.walk(IN):
    if all(c in dirs for c in CLASSES):
        CLIPS = root
        break

print('code zip :', CODE_ZIP)
print('ball wts :', BALL_W)
print('clips    :', CLIPS)
assert CODE_ZIP and BALL_W and CLIPS, 'One of the three inputs is missing - check Add Input'

CODE = f'{WORK}/fyp'
shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(CODE)
sys.path.insert(0, CODE)

n = sum(len(glob.glob(f'{CLIPS}/{c}/*')) for c in CLASSES)
for c in CLASSES:
    print(f'  {c:<22}: {len(glob.glob(f"{CLIPS}/{c}/*")):>4} clips')
print('total clips:', n)

# teams.py / preflight.py must be present, or the zip is the stale one.
for must in ('teams.py', 'preflight.py'):
    assert os.path.exists(f'{CODE}/{must}'), (
        f'{must} missing -> you uploaded the OLD fyp_code.zip. Rebuild and re-upload.')
print('code OK: teams.py + preflight.py present')

## 3. Test suite gate

Runs before anything expensive. A broken import here costs seconds; the same break discovered 40 minutes into extraction is what §11.7 and §12.3 were about.

In [ ]:
!cd {CODE} && python -m pytest tests -q 2>&1 | tail -15

## 4. Detector sanity check on the real clips

This is §13.1 re-run on the GPU, on the actual clips about to be extracted. **Do not skip it.** If the player count here is near zero, extraction will silently produce 508 CSVs of empty formations - which is exactly what happened before.

In [ ]:
import numpy as np, cv2, random
from ultralytics import YOLO

stock  = YOLO('yolo11n.pt')
custom = YOLO(BALL_W)
S_PERS = [k for k, v in stock.names.items() if v == 'person'][0]
C_BALL = [k for k, v in custom.names.items() if v == 'ball'][0]
C_PLAY = [k for k, v in custom.names.items() if v == 'player'][0]

random.seed(0)
sample = []
for c in CLASSES:
    sample += random.sample(glob.glob(f'{CLIPS}/{c}/*'), 5)

rows = []
for clip in sample:
    cap = cv2.VideoCapture(clip)
    ok, fr = cap.read()
    cap.release()
    if not ok:
        continue
    r_s = stock.predict(fr, imgsz=1280, conf=0.25, verbose=False)[0]
    r_c = custom.predict(fr, imgsz=1280, conf=0.25, verbose=False)[0]
    cs = r_s.boxes.cls.cpu().numpy().astype(int)
    cc = r_c.boxes.cls.cpu().numpy().astype(int)
    rows.append(((cs == S_PERS).sum(), (cc == C_PLAY).sum(), (cc == C_BALL).any()))

a = np.array(rows, dtype=float)
print(f'{len(a)} clips sampled (first frame each)')
print(f'  stock  yolo11n players/frame : mean {a[:,0].mean():5.1f}  median {np.median(a[:,0]):.0f}')
print(f'  custom fine-tune players/frame: mean {a[:,1].mean():5.1f}  median {np.median(a[:,1]):.0f}')
print(f'  custom ball recall            : {100*a[:,2].mean():.0f}%')
assert np.median(a[:,0]) >= 4, 'stock detector is not seeing players - stop and investigate'
print('\nOK -> players come from stock, ball comes from the fine-tune.')

## 5. Re-extract all 508 clips

These flags **must** stay identical to the ones `pipeline.py` uses at inference time. That is the whole point of the retrain; a mismatch here silently recreates the bug being fixed.

`--auto-court` is load-bearing. Verified locally on six clips: **without it, 1 clip in 4 extracted with 0.0 players/frame** (the crowd captured the colour clusters); **with it, all six came out at 5.2-6.0 players/frame** - essentially the exact team size - and ball recall 83-100%.

Roughly 20-45 min on a T4 (vs 2-4 h on your laptop CPU). Two detector passes per frame at imgsz 1280.

In [ ]:
OUT_CSV = f'{WORK}/training_csv_v3'

# Every flag here must match MAKE_DEMO_VIDEO.bat exactly. --auto-court is not
# optional: without the court mask the crowd enters the population, captures the
# jersey-colour clusters, and clips extract with ZERO players. Measured on
# videoplayback (1) clip_002: 0.0 players/frame without it, 5.9 with it.
cmd = (
    f'cd {CODE} && python prepare_training_data.py "{CLIPS}" '
    f'--output-dir {OUT_CSV} '
    f'--yolo-model yolo11n.pt '
    f'--ball-model "{BALL_W}" '
    f'--imgsz 1280 '
    f'--team-split colour '
    f'--auto-court --court-coords linear '
    f'--tracker botsort '
    f'--clean-output 2>&1 | grep -vE "requirements:|notice|Requirement already" | tail -30'
)
!{cmd}


## 6. Audit the extracted CSVs before training on them

`diagnose_data.py` prints the Section 1 health table. The number that matters most: how many clips came out with a usable formation rather than empty slots.

In [ ]:
import pandas as pd

csvs = sorted(glob.glob(f'{OUT_CSV}/*.csv'))
print(f'{len(csvs)} CSVs extracted\n')

pcols = [f'p{i}_{ax}' for i in range(1, 7) for ax in ('x', 'y')]
present, spacing_ok = [], 0
for f in csvs:
    d = pd.read_csv(f)
    have = [c for c in pcols if c in d.columns]
    xs = d[have].replace(0.0, np.nan)
    present.append(xs.notna().sum(axis=1).mean() / 2.0)

present = np.array(present)
print(f'players present per frame : mean {present.mean():.2f}  median {np.median(present):.2f}')
print(f'clips with >=3 players    : {100*(present>=3).mean():.0f}%')
print(f'clips with <1 player      : {100*(present<1).mean():.0f}%   <- must be near 0')

if (present < 1).mean() > 0.15:
    print('\n[STOP] Too many empty clips. Do NOT train on this - re-check cell 4.')
else:
    print('\nExtraction looks healthy. Proceed to training.')

!cd {CODE} && python diagnose_data.py {OUT_CSV} 2>&1 | tail -25

## 7. Train the Mamba classifier

Random split first - this is the number comparable to the old 63.6%.

In [ ]:
!cd {CODE} && python train_mamba.py {OUT_CSV} \
    --augment --epochs 80 \
    --val_split 0.2 --test_split 0.1 \
    --checkpoint {WORK}/mamba_checkpoint_v3.pt \
    --history_csv {WORK}/training_history_v3.csv 2>&1 | tail -40

## 8. Leave-one-video-out - the honest cross-video number

Sec 12.10: the clip-random split shares source videos between train and test, so it measures *within-video* generalisation. LOVO holds out every clip of one source video.

There are **four** source videos: `(3)` 201 clips, `(plain)` 130, `(1)` 126, `(4)` 51. Video `(4)` is the one the fine-tune could not see, so its clips extracted near-empty before this fix - that fold is newly meaningful.

**Expect these below the cell-7 number - that gap IS the finding.** Reporting both, with the gap explained, is what makes the evaluation chapter defensible.

In [ ]:
# Four source videos, not three: (3)=201 clips, (plain)=130, (1)=126, (4)=51.
# Video (4) is the one the fine-tune was blind to, so its 51 clips extracted
# near-empty before this fix - that fold is newly meaningful.
FOLDS = ['(1)', '(3)', '(4)', '(plain)']
KEEP = 'LOVO_RESULT|accuracy|macro|balanced|recall|Traceback|Error'

for fold in FOLDS:
    tag = fold.strip('()')
    print('=' * 62)
    print('LOVO fold - holding out video', fold)
    print('=' * 62)
    cmd = (
        f'cd {CODE} && python train_mamba.py {OUT_CSV} '
        f'--augment --epochs 80 --test-video "{fold}" '
        f'--checkpoint {WORK}/mamba_lovo_{tag}.pt '
        f'2>&1 | tee {WORK}/lovo_{tag}.log | grep -E "{KEEP}" | tail -14'
    )
    !{cmd}


In [ ]:
# Collect every LOVO_RESULT line into one table for the report.
import re, glob

pat = re.compile(r'LOVO_RESULT video=(\S+) acc=([\d.]+) macro_f1=([\d.]+) balanced_acc=([\d.]+)')

print(f"{'fold':<10} {'acc':>7} {'macroF1':>9} {'balAcc':>8}")
print('-' * 38)
for log in sorted(glob.glob(f'{WORK}/lovo_*.log')):
    m = pat.search(open(log, errors='ignore').read())
    tag = log.split('lovo_')[-1][:-4]
    if m:
        print(f'{m.group(1):<10} {float(m.group(2)):>7.3f} {float(m.group(3)):>9.3f} {float(m.group(4)):>8.3f}')
    else:
        print(f'{tag:<10}   no LOVO_RESULT line - open the log')


## 9. Role analysis + package the outputs

Everything in `/kaggle/working` is downloadable from the notebook's Output tab when the session ends.

In [ ]:
!cd {CODE} && python analytics.py {OUT_CSV} {WORK}/coordination_analysis_v3.csv 2>&1 | tail -20
!cd {CODE} && python role_analysis.py {OUT_CSV} {WORK}/role_analysis_v3.csv 2>&1 | tail -20
!cd {CODE} && python infer_mamba.py {WORK}/mamba_checkpoint_v3.pt {OUT_CSV} \
    --output_csv {WORK}/preds_v3.csv 2>&1 | tail -25

In [ ]:
import shutil, os

shutil.make_archive(f'{WORK}/training_csv_v3', 'zip', OUT_CSV)

print('Download these from the Output tab:\n')
for f in sorted(glob.glob(f'{WORK}/*')):
    if os.path.isfile(f):
        print(f'  {os.path.basename(f):<34} {os.path.getsize(f)/1e6:8.2f} MB')

print('\nMinimum set to bring home:')
print('  mamba_checkpoint_v3.pt      -> project root (replaces mamba_checkpoint_v2.pt)')
print('  training_csv_v3.zip         -> unzip to project root, for cheap local retraining')
print('  training_history_v3.csv     -> training curves for the report')
print('  coordination_analysis_v3.csv / role_analysis_v3.csv -> analytics chapter')
print('\nThen render the demo locally with the MATCHING flags:')
print('  MAKE_DEMO_VIDEO.bat   (already uses them; edit the checkpoint name to v3)')